# QIntent Structured Semantic Score for qLDPC-Style Post-Decoding

This notebook demonstrates a public-preview QIntent workflow for ranking qLDPC-style correction candidates after a decoder has produced candidate hypotheses.

The goal is not to claim that QIntent replaces a quantum error-correction decoder. Instead, this demo shows how QDSV can represent post-decoding candidates as structured decision states and rank them using prepared evidence blocks such as syndrome support, logical safety, decoder confidence, propagation safety, and risk.

The internal QDSV scoring formula is not exposed. The notebook only exposes the public declaration, evidence fields, block-level trace, and ranking output.

## 1. Install the SDK from GitHub main

The structured semantic score operation is installed from the public GitHub repository so this notebook can run before the next PyPI release.

In [ ]:
!pip install -q --upgrade "git+https://github.com/qdsvquantum-afk/qintent.git"

In [ ]:
import json
from pathlib import Path

import pandas as pd

from qintent import QIntentClient

API_URL = "https://qdsv-api-568788785178.us-central1.run.app/api"
client = QIntentClient(api_url=API_URL, timeout=60)

spec = client.spec()
spec["status"]

## 2. Generate qLDPC-style candidate evidence from sparse checks

This section builds a small public-safe CSS/qLDPC-style toy decoding problem. It defines a sparse check matrix through per-qubit syndrome columns, chooses an observed syndrome, enumerates low-weight correction hypotheses, and converts each candidate into prepared 0..1000 evidence signals.

This is still a toy experiment, not a full qLDPC benchmark. The improvement over the previous version is that the rows are no longer hand-written candidates: they are generated from a sparse check structure and a syndrome-matching candidate generator.


In [ ]:
from itertools import combinations

# Sparse toy check structure. Each qubit column indicates which checks would fire.
# This plays the role of a small CSS/qLDPC-style syndrome map for public-safe testing.
check_columns = {
    0: (1, 0, 1, 0),
    1: (1, 1, 0, 0),  # high-confidence but logical-risky single-qubit hypothesis
    2: (1, 0, 0, 0),
    3: (0, 1, 0, 0),
    4: (0, 0, 1, 1),
    5: (0, 1, 0, 1),
    6: (1, 0, 1, 1),
}

observed_syndrome = (1, 1, 0, 0)
logical_sensitive_qubits = {1, 6}
max_candidate_weight = 3


def xor_syndrome(qubits):
    out = [0] * len(observed_syndrome)
    for q in qubits:
        out = [a ^ b for a, b in zip(out, check_columns[q])]
    return tuple(out)


def hamming_distance(a, b):
    return sum(x != y for x, y in zip(a, b))


def generate_candidates():
    generated = []
    candidate_index = 0
    for weight in range(1, max_candidate_weight + 1):
        for qubits in combinations(check_columns.keys(), weight):
            predicted = xor_syndrome(qubits)
            mismatch = hamming_distance(predicted, observed_syndrome)
            if mismatch != 0:
                continue

            logical_overlap = sum(q in logical_sensitive_qubits for q in qubits)

            # Prepared public signals. These are intentionally visible operational features,
            # not the private QDSV scoring formula.
            generated.append({
                "candidate_index": candidate_index,
                "correction_qubits": "".join(str(q) for q in qubits),
                "candidate_weight": weight,
                "predicted_syndrome": "".join(str(bit) for bit in predicted),
                "syndrome_support": max(0, 1000 - 180 * mismatch - 25 * abs(weight - 1)),
                "check_consistency": max(0, 1000 - 250 * mismatch),
                "logical_preservation": max(0, 960 - 280 * logical_overlap - 30 * max(0, weight - 1)),
                "distance_safety": max(0, 940 - 220 * logical_overlap - 25 * weight),
                "decoder_confidence": max(0, 1000 - 90 * weight - 120 * mismatch),
                "propagation_safety": max(0, 930 - 70 * weight - 80 * logical_overlap),
                "syndrome_risk": min(1000, 40 + 120 * mismatch + 10 * weight),
                "logical_risk": min(1000, 30 + 230 * logical_overlap + 25 * weight),
                "syndrome_entropy_adjustment": 20 if weight == 2 and logical_overlap == 0 else (-15 if logical_overlap else 0),
            })
            candidate_index += 1
    return generated


rows = generate_candidates()
df = pd.DataFrame(rows)

print(f"Generated {len(rows)} syndrome-compatible candidate corrections")
df


## 3. Baseline: decoder-confidence-only ranking

A simple post-decoding policy could choose the candidate with the highest decoder confidence. This baseline ignores logical risk, distance safety, propagation safety, and syndrome-level adjustments.

In [ ]:
baseline = df.sort_values("decoder_confidence", ascending=False).reset_index(drop=True)
baseline[["candidate_index", "decoder_confidence", "logical_risk", "logical_preservation", "distance_safety"]]

## 4. QIntent structured semantic score declaration

The QIntent source below declares public evidence blocks. It does not expose the private QDSV formula.

The three blocks are:

- `syndrome`: syndrome support and check consistency.
- `logical_safety`: logical preservation and distance safety.
- `decoder`: decoder confidence and propagation safety.

In [ ]:
source = """
find_rows("candidate_index")
  .using_structured_semantic_score([
      block("syndrome", [
          signal("syndrome_support", influence=30, priority=2),
          signal("check_consistency", influence=20, priority=1),
      ], influence=30, priority=2, risk_adjustment="syndrome_risk", adjustments=[
          adjustment("syndrome_entropy_adjustment", influence=5),
      ]),
      block("logical_safety", [
          signal("logical_preservation", influence=40, priority=3),
          signal("distance_safety", influence=20, priority=2),
      ], influence=40, priority=3, risk_adjustment="logical_risk"),
      block("decoder", [
          signal("decoder_confidence", influence=25, priority=1),
          signal("propagation_safety", influence=15, priority=2),
      ], influence=30, priority=1),
  ], global_risk="logical_risk", profile="qldpc_post_decoding")
  .accept_if(threshold=600)
  .rank()
  .top_k(2)
"""

print(source)

## 5. Explain before running

`explain()` returns a semantic execution passport. For this operation, it should identify structured semantic scoring, block count, signal count, and confirm that the internal formula is not exposed.

In [ ]:
passport = client.explain(source, rows=rows)
predicate_summary = passport["semantic_execution_passport"]["predicate"]
predicate_summary

Expected public evidence:

- `kind`: `structured_semantic_score`
- `blocks_count`: 3
- `signals_count`: 6
- `internal_formula_exposed`: `False`

In [ ]:
assert predicate_summary["kind"] == "structured_semantic_score"
assert predicate_summary["blocks_count"] == 3
assert predicate_summary["signals_count"] == 6
assert predicate_summary["internal_formula_exposed"] is False

predicate_summary

## 6. Run QIntent and inspect ranked correction candidates

In [ ]:
result = client.run(source, rows=rows)
result["status"]

In [ ]:
selected_rows = result["result"]["selected_rows"]
ranked = pd.DataFrame(selected_rows)
ranked[[
    "candidate_index",
    "correction_qubits",
    "candidate_weight",
    "_qdsv_rank",
    "_qdsv_structured_semantic_score",
    "decoder_confidence",
    "logical_preservation",
    "distance_safety",
    "logical_risk",
]]


## 7. Compare baseline vs QDSV structured ranking

The baseline chooses by decoder confidence alone. QDSV ranks candidates using structured evidence and risk. In this toy example, the top QDSV candidate is not the same as the highest-confidence decoder candidate.

In [ ]:
comparison = pd.DataFrame([
    {
        "method": "decoder_confidence_only",
        "top_candidate": int(baseline.iloc[0]["candidate_index"]),
        "correction_qubits": baseline.iloc[0]["correction_qubits"],
        "candidate_weight": int(baseline.iloc[0]["candidate_weight"]),
        "decoder_confidence": int(baseline.iloc[0]["decoder_confidence"]),
        "logical_risk": int(baseline.iloc[0]["logical_risk"]),
    },
    {
        "method": "qdsv_structured_semantic_score",
        "top_candidate": int(ranked.iloc[0]["candidate_index"]),
        "correction_qubits": ranked.iloc[0]["correction_qubits"],
        "candidate_weight": int(ranked.iloc[0]["candidate_weight"]),
        "decoder_confidence": int(ranked.iloc[0]["decoder_confidence"]),
        "logical_risk": int(ranked.iloc[0]["logical_risk"]),
    },
])

comparison


## 8. Inspect the audit trace for the top candidate

The selected row includes a public audit trace with block-level values, risk fields, selected status, and rank. The private formula remains hidden.

In [ ]:
top_trace = selected_rows[0]["_qdsv_decision_model"]
print(json.dumps(top_trace, indent=2))

## 9. Save evidence artifacts

The JSON evidence file can be attached to a review pack, reproduced later, or compared with future experiments using real qLDPC decoders.

In [ ]:
evidence = {
    "api_url": API_URL,
    "profile": "qldpc_post_decoding",
    "toy_sparse_check_structure": {
        "check_columns": {str(k): list(v) for k, v in check_columns.items()},
        "observed_syndrome": list(observed_syndrome),
        "logical_sensitive_qubits": sorted(logical_sensitive_qubits),
        "max_candidate_weight": max_candidate_weight,
        "candidate_generation": "enumerate syndrome-compatible low-weight correction hypotheses",
    },
    "rows": rows,
    "qintent_source": source,
    "predicate_summary": predicate_summary,
    "baseline_top_candidate": int(baseline.iloc[0]["candidate_index"]),
    "qdsv_top_candidate": int(ranked.iloc[0]["candidate_index"]),
    "comparison": comparison.to_dict(orient="records"),
    "selected_rows": selected_rows,
    "internal_formula_exposed": False,
}

Path("qdsv_qldpc_structured_score_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
ranked.to_csv("qdsv_qldpc_structured_score_selected_rows.csv", index=False)
comparison.to_csv("qdsv_qldpc_structured_score_comparison.csv", index=False)

print("Saved:")
print("- qdsv_qldpc_structured_score_evidence.json")
print("- qdsv_qldpc_structured_score_selected_rows.csv")
print("- qdsv_qldpc_structured_score_comparison.csv")


## Interpretation

This notebook demonstrates the post-decoding role of QDSV/QIntent: given correction hypotheses generated from a sparse check structure and an observed syndrome, QDSV represents the candidates as structured decision states and QIntent executes a public structured scoring declaration over prepared evidence.

The baseline ranks by decoder confidence alone. QDSV can select a lower-confidence candidate when its syndrome evidence, logical safety, propagation safety and risk profile make it a safer correction hypothesis.

The next experimental step is to replace this toy sparse-check generator with candidates produced by an actual qLDPC decoder or predecoder and evaluate whether QDSV improves ranking stability, logical-risk avoidance, auditability, or ambiguity resolution.
